[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/08_knn/01_intuition_and_distances.ipynb)

# Bài 01 — K-Nearest Neighbors: trực giác và khoảng cách

**🎯 Mục tiêu**
- Hiểu ý tưởng "hỏi hàng xóm": dự đoán bằng nhãn của $k$ điểm gần nhất.
- Các thước đo khoảng cách: Euclid, Manhattan, Minkowski, cosine, Hamming — và khi nào dùng cái nào.
- Thấy decision boundary của KNN thay đổi theo $k$; Voronoi với $k = 1$.
- Hiểu KNN là **lazy learner** (học = ghi nhớ) và **non-parametric**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/08_knn

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import make_classification_2d
from ml_course.visualization import PALETTE, plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ý tưởng: "Hãy cho tôi biết hàng xóm của bạn là ai..."

Để dự đoán cho một điểm mới $\mathbf x$:
1. Tính khoảng cách từ $\mathbf x$ tới **mọi** điểm trong tập train.
2. Lấy $k$ điểm gần nhất — **láng giềng** $\mathcal N_k(\mathbf x)$.
3. **Phân loại:** bỏ phiếu đa số nhãn của $k$ láng giềng. **Hồi quy:** lấy trung bình giá trị của chúng.

$$\hat y = \arg\max_c \sum_{i \in \mathcal N_k(\mathbf x)} \mathbb 1[y^{(i)} = c] \qquad\text{hoặc}\qquad \hat y = \frac1k\sum_{i \in \mathcal N_k(\mathbf x)} y^{(i)}$$

Không có tham số nào được "học" — KNN chỉ **ghi nhớ** dữ liệu train (*lazy learning*), mọi tính toán dồn vào lúc dự đoán. Số "tham số" thực chất tăng theo kích thước dữ liệu → KNN là model **non-parametric**.

In [ ]:
X, y = make_classification_2d("blobs", n=60, noise=0.3, seed=3)
query = np.array([0.2, 0.3])
dist = np.linalg.norm(X - query, axis=1)
k = 5
nn = np.argsort(dist)[:k]

fig, ax = plt.subplots(figsize=(6.5, 6))
for c in [0, 1]:
    ax.scatter(X[y == c, 0], X[y == c, 1], color=PALETTE[c], s=40, label=f"lớp {c}")
ax.scatter(*query, marker="*", s=350, color="black", label="điểm cần dự đoán")
for i in nn:
    ax.plot([query[0], X[i, 0]], [query[1], X[i, 1]], "k--", lw=0.8)
ax.add_patch(plt.Circle(query, dist[nn[-1]], fill=False, color="gray", ls=":"))
ax.set_aspect("equal")
votes = np.bincount(y[nn], minlength=2)
ax.set_title(f"k = {k}: phiếu bầu lớp 0 = {votes[0]}, lớp 1 = {votes[1]} → dự đoán lớp {np.argmax(votes)}")
ax.legend(loc="lower right")
plt.show()

## 2. Thước đo khoảng cách

| Khoảng cách | Công thức | Dùng khi |
|---|---|---|
| **Euclid** (L2) | $\sqrt{\sum_j (a_j - b_j)^2}$ | mặc định, feature số đã chuẩn hoá |
| **Manhattan** (L1) | $\sum_j \lvert a_j - b_j\rvert$ | ít nhạy với chênh lệch lớn ở một chiều; dữ liệu dạng lưới |
| **Minkowski** ($L_p$) | $\left(\sum_j \lvert a_j - b_j\rvert^p\right)^{1/p}$ | tổng quát hoá: $p = 1$ Manhattan, $p = 2$ Euclid |
| **Cosine** | $1 - \frac{\mathbf a^\top\mathbf b}{\lVert\mathbf a\rVert\lVert\mathbf b\rVert}$ | văn bản (TF-IDF), embedding — quan tâm **hướng**, bỏ qua độ dài |
| **Hamming** | số vị trí khác nhau | feature nhị phân / phân loại |

In [ ]:
from ml_course.algorithms.knn import pairwise_distances

a = np.array([[1.0, 2.0]])
b = np.array([[4.0, 6.0]])
for metric in ["euclidean", "manhattan", "cosine"]:
    print(f"{metric:10s}: {pairwise_distances(a, b, metric)[0, 0]:.4f}")
print(f"minkowski p=3: {pairwise_distances(a, b, 'minkowski', p=3)[0, 0]:.4f}")

**Hình dạng "hình tròn đơn vị"** của mỗi metric quyết định "láng giềng" trông như thế nào — tập các điểm cách gốc đúng 1 đơn vị:

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
fig, ax = plt.subplots(figsize=(5, 5))
for p, label in [(1, "Manhattan (p=1)"), (2, "Euclid (p=2)"), (4, "p=4"), (np.inf, "Chebyshev (p=∞)")]:
    pts = np.c_[np.cos(theta), np.sin(theta)]
    pts /= np.linalg.norm(pts, ord=p, axis=1, keepdims=True)
    ax.plot(pts[:, 0], pts[:, 1], label=label)
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("Các điểm cách gốc toạ độ 1 đơn vị")
plt.show()

**Cosine vs Euclid** — hai văn bản về cùng chủ đề nhưng một văn bản dài gấp 3 lần (vector đếm từ gấp 3): Euclid coi chúng xa nhau, cosine coi chúng giống hệt.

In [ ]:
doc_short = np.array([[2.0, 1.0, 0.0]])        # đếm từ: "bóng đá", "trận", "chứng khoán"
doc_long = 3 * doc_short
doc_other = np.array([[0.0, 1.0, 2.0]])
for name, other in [("văn bản dài cùng chủ đề", doc_long), ("văn bản chủ đề khác", doc_other)]:
    print(f"{name:25s} Euclid = {pairwise_distances(doc_short, other)[0, 0]:.2f} | cosine distance = {pairwise_distances(doc_short, other, 'cosine')[0, 0]:.2f}")

## 3. Decision boundary thay đổi theo k

- $k = 1$: mỗi điểm train "sở hữu" vùng không gian gần nó nhất → **sơ đồ Voronoi**; ranh giới lởm chởm, bám theo nhiễu → **overfit** (lỗi train = 0!).
- $k$ lớn: ranh giới mượt; $k = n$: luôn dự đoán lớp đa số → **underfit**.

$k$ chính là "núm vặn" bias–variance của KNN (chương 05, bài 04): $k$ nhỏ → variance cao, $k$ lớn → bias cao.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X, y = make_classification_2d("moons", n=250, noise=0.3, seed=1)
X_val, y_val = make_classification_2d("moons", n=2000, noise=0.3, seed=2)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, k in zip(axes, [1, 5, 25, 125]):
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)
    plot_decision_boundary(knn, X, y, ax=ax)
    ax.set_title(f"k = {k}: train {knn.score(X, y):.2f} | validation {knn.score(X_val, y_val):.2f}")
plt.show()

## 4. KNN cho hồi quy

Dự đoán = trung bình $y$ của $k$ láng giềng → đường dự đoán dạng **bậc thang**; $k$ lớn → mượt hơn nhưng bị "kéo phẳng" ở các vùng cong và ở hai đầu.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

from ml_course.datasets import make_regression_1d

Xr, yr = make_regression_1d(n=80, kind="sin", noise=0.5, seed=0)
grid = np.linspace(-3.2, 3.2, 400).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(17, 3.8))
for ax, k in zip(axes, [1, 7, 40]):
    reg = KNeighborsRegressor(n_neighbors=k).fit(Xr, yr)
    ax.scatter(Xr, yr, s=12)
    ax.plot(grid, reg.predict(grid), "r", lw=2)
    ax.set_title(f"KNN regression, k = {k}")
plt.show()

> 💡 KNN **không ngoại suy** được: ngoài vùng dữ liệu, dự đoán bằng trung bình các điểm ở rìa (đường phẳng ở hai đầu).

## 🧠 Kiểm tra nhanh

1. Vì sao gọi KNN là "lazy learner"? Hệ quả về thời gian train và thời gian dự đoán?
2. Với $k = 1$, accuracy trên tập train là bao nhiêu? Vì sao điều đó không đáng tin?
3. Khi nào nên dùng cosine thay cho Euclid?
4. Tăng $k$ ảnh hưởng thế nào tới bias và variance?

## 📌 Tóm tắt

- KNN: dự đoán bằng bỏ phiếu (phân loại) hoặc trung bình (hồi quy) của $k$ điểm gần nhất.
- Lazy + non-parametric: không học tham số, ghi nhớ toàn bộ dữ liệu.
- Kết quả phụ thuộc mạnh vào **thước đo khoảng cách** và **thang đo feature** (bài 03).
- $k$ điều khiển bias–variance.

## 📚 Tài liệu tham khảo

- Cover & Hart (1967) — *Nearest neighbor pattern classification*
- Vũ Hữu Tiệp — [K-nearest neighbors](https://machinelearningcoban.com/2017/01/08/knn/)
- ESL §13.3